<a href="https://colab.research.google.com/github/Aljwharah-h/SDAIA-AI-Agents-System-program/blob/main/Assignment4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Set up

In [1]:
%pip install langchain-community pypdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 332.2/332.2 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 503.5/503.5 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.7/64.7 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 1.7 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.18
    Uninstalling langchain-core-1.2.18:
      Successfully uninstalled langchain-core-1.2.18
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
goo

Loading documents

In [2]:
from langchain_core.documents import Document
from langchain_community.document_loaders import PyPDFLoader

file_path = "https://files.eric.ed.gov/fulltext/ED536788.pdf"
loader = PyPDFLoader(file_path)

docs = loader.load()

print(len(docs))

112


In [3]:
print(f"{docs[0].page_content[:200]}\n")
print(docs[0])

Introduction to  
Data Analysis 
Handbook
Migrant & Seasonal Head Start 
T echnical Assistance Center
Academy for Educational Development
“If 
I knew what 
you were going to use 
the information for  

page_content='Introduction to  
Data Analysis 
Handbook
Migrant & Seasonal Head Start 
T echnical Assistance Center
Academy for Educational Development
“If 
I knew what 
you were going to use 
the information for  
I would have done a better 
job of collecting it.”
--Famous quote from a Migrant and 
Seasonal Head Start (MSHS) staff 
person to MSHS director at a 
Community Assessment 
Training.' metadata={'producer': 'Adobe PDF Library 7.0', 'creator': 'Adobe InDesign CS2 (4.0.2)', 'creationdate': '2006-08-07T18:30:15-06:00', 'moddate': '2006-08-07T18:30:39-06:00', 'trapped': '/False', 'source': 'https://files.eric.ed.gov/fulltext/ED536788.pdf', 'total_pages': 112, 'page': 0, 'page_label': 'i'}


Splitting

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000, chunk_overlap=200, add_start_index=True
)
all_splits = text_splitter.split_documents(docs)

print(len(all_splits))

192


Embeddings

In [5]:
%pip install langchain-huggingface sentence-transformers

In [6]:
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
vector_1 = embeddings.embed_query(all_splits[0].page_content)
vector_2 = embeddings.embed_query(all_splits[1].page_content)

assert len(vector_1) == len(vector_2)
print(f"Generated vectors of length {len(vector_1)}\n")
print(vector_1[:10])

Generated vectors of length 768

[-0.06831696629524231, 0.0007002928759902716, -0.04274025186896324, -0.042280178517103195, -0.022585870698094368, 0.05837533622980118, 0.04190812259912491, -0.036462098360061646, 0.048494499176740646, -0.001817591954022646]


Vector stores

In [8]:
from langchain_core.vectorstores import InMemoryVectorStore

vector_store = InMemoryVectorStore(embeddings)

In [9]:
ids = vector_store.add_documents(all_splits)
print(ids)

['7ae24b2f-d12f-4c5c-8516-9e10ef10224f', '81f2eea1-b601-4bab-9da3-2de79d20269f', 'b444daa6-fa10-4d8e-b2b6-4d4545c6f89e', 'edfa7382-35ca-4ee3-93b7-cf028fa7ac75', '0c92cfcc-c111-4f47-a834-315d37e3b1f9', '0cd414c9-420d-4be7-9942-bd6d8b36f071', '49204e5f-f330-49c3-aac8-c2a50dc57351', 'f3e2a4aa-2cab-497a-818f-9a0d2be9c7a4', 'ef502580-ece9-4732-93f3-cdee09ae6290', '69d86175-2f28-4f88-a4f3-a0d4cd1ed133', '7d2e219b-7482-4caa-a429-172fdc9ebf02', '1688ab5a-fb7f-459f-8df5-e758eefa1659', '7c880f90-a859-4201-82b4-82b988870cdb', 'd91db764-f457-4030-b5c3-226ff84b5c74', 'fe10aaea-44b6-4350-80ed-0e261e0d8ac5', '737764fe-f326-4472-9547-90ffbb0ab254', '0296274f-fb52-441e-9df3-b592f8c59940', 'dfe396cf-b58d-4a1d-b5d5-45c58493a855', 'f8c81058-e898-4165-800e-fbf38bc607b5', '27106f31-f256-462d-9a91-ca80ecd7f991', 'f67c175a-83e3-40b6-886f-693d805a0a08', '1f3eee59-1d7b-410b-9f70-a965f5e43450', '1cea2738-4de2-47cd-b573-9e4ea9500e78', 'ee85f567-70b4-419c-9b90-ae7f8a3b3863', '7a96e9a9-a003-464f-b605-7eb1bc3d55c4',

In [10]:
results = vector_store.similarity_search(
    "What does this document talk about??"
)

print(results[0])

page_content='Sample Planning Table
Purpose Questions
Data 
Collection 
Methods
Needed 
Resources Lead Person Timeframe
Purpose 1
Purpose 2
Purpose 
The planning piece of the framework gives you an opportunity to identify the resources 
that you will need. Some examples of needed resources include the translation of key 
pieces of information for parents, additional clerical support and scheduling meetings in 
conjunction with other activities.
As with other planning pieces within Head Start it is important that you consult with 
the Policy Council (PC) regarding the plan prior to its implementation. Solicit ideas and 
assistance from PC members regarding the proposed data analysis process.
Important Tip: Traditionally many programs find themselves in a crunch when dealing with 
data analysis.  Taking time to plan out your process will save you time in the long run.' metadata={'producer': 'Adobe PDF Library 7.0', 'creator': 'Adobe InDesign CS2 (4.0.2)', 'creationdate': '2006-08-07T18:

In [11]:
# Note that providers implement different scores; the score here
# is a distance metric that varies inversely with similarity.

results = vector_store.similarity_search_with_score("How does the Transformer encoder work?")
doc, score = results[0]
print(f"Score: {score}\n")
print(doc)

Score: 0.19506011590001104

page_content='© AED/TAC-12 Spring 2006. 0' metadata={'producer': 'Adobe PDF Library 7.0', 'creator': 'Adobe InDesign CS2 (4.0.2)', 'creationdate': '2006-08-07T18:30:15-06:00', 'moddate': '2006-08-07T18:30:39-06:00', 'trapped': '/False', 'source': 'https://files.eric.ed.gov/fulltext/ED536788.pdf', 'total_pages': 112, 'page': 95, 'page_label': '90', 'start_index': 0}


In [12]:
embedding = embeddings.embed_query("What are the advantages of the Transformer over RNNs?")

results = vector_store.similarity_search_by_vector(embedding)
print(results[0])

page_content='© AED/TAC-12 Spring 2006. 6
Appendix C: Supplemental                                          Resources' metadata={'producer': 'Adobe PDF Library 7.0', 'creator': 'Adobe InDesign CS2 (4.0.2)', 'creationdate': '2006-08-07T18:30:15-06:00', 'moddate': '2006-08-07T18:30:39-06:00', 'trapped': '/False', 'source': 'https://files.eric.ed.gov/fulltext/ED536788.pdf', 'total_pages': 112, 'page': 101, 'page_label': '96', 'start_index': 0}


Retrievers

In [14]:
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 1},
)

retriever.batch(
    [
        "What is the main idea proposed in the pdf?",
        "What is data analysis?",

    ],
)

[[Document(id='e1d48e44-7bbd-4c70-9cb2-575e3da51fb7', metadata={'producer': 'Adobe PDF Library 7.0', 'creator': 'Adobe InDesign CS2 (4.0.2)', 'creationdate': '2006-08-07T18:30:15-06:00', 'moddate': '2006-08-07T18:30:39-06:00', 'trapped': '/False', 'source': 'https://files.eric.ed.gov/fulltext/ED536788.pdf', 'total_pages': 112, 'page': 26, 'page_label': '21', 'start_index': 1505}, page_content='members disagree over the interpretation of the data or was there consensus?  \nWriting, Reporting, and Dissemination. How well did the writing tell the story of the \ndata?  Did the intended audience find the presentation of information effective?\nSee Appendix C for a checklist on how to evaluate your data analysis process.  Section \nV walks you through several content examples using all of the process components \nidentified above. \nin sum, data analysis is a process: a series of connected activities designed to obtain \nmeaningful information from data that have been collected. As the graph